# 26 — Report Assets: Pre-Adam Briefing v1

This notebook converts the corrected pre-Adam pack into report-ready assets.

It assumes the current model position is:

- caveat recode fixed through 22b;
- SDP validation fixed through 24b5;
- Yorkshire case study fixed through 24c2;
- pre-Adam revision pack fixed through 25d.

The output is an asset pack for a professional report, not another modelling pass.

## 26.1 Setup and paths

Expected project structure:

```text
Electoral_Tribes/
  data/
    processed/
      pre_adam_report_revision_v4/      # or equivalent folder containing pre_adam_* outputs
      report_assets_pre_adam_v1/
    geography/
      boundaries/                       # optional; WD25 ward boundaries for maps
  notebooks/
```

The notebook searches recursively under `data/processed` for the required `pre_adam_*` files, so exact folder names can vary.

In [84]:
from pathlib import Path
from datetime import datetime
import textwrap
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

warnings.filterwarnings("ignore")

NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name.lower() == "notebooks" else NOTEBOOK_DIR

DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
GEOGRAPHY_DIR = DATA_DIR / "geography"
BOUNDARY_DIR = GEOGRAPHY_DIR

OUTPUT_DIR = PROCESSED_DIR / "report_assets_pre_adam_v2"
TABLE_DIR = OUTPUT_DIR / "tables"
CHART_DIR = OUTPUT_DIR / "charts"
MAP_DIR = OUTPUT_DIR / "maps"
APPENDIX_DIR = OUTPUT_DIR / "appendices"
TEXT_DIR = OUTPUT_DIR / "text"
MANIFEST_DIR = OUTPUT_DIR / "manifest"

for d in [OUTPUT_DIR, TABLE_DIR, CHART_DIR, MAP_DIR, APPENDIX_DIR, TEXT_DIR, MANIFEST_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)
print("Processed:", PROCESSED_DIR)
print("Output:", OUTPUT_DIR)

Project: c:\Users\keena\Documents\Electoral_Tribes
Processed: c:\Users\keena\Documents\Electoral_Tribes\data\processed
Output: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_pre_adam_v2


## 26.2 Report style configuration

In [85]:
REPORT_VERSION = "Pre-Adam v1"
REPORT_DATE = datetime.today().strftime("%Y-%m-%d")

NAVY = "#112A46"
MID_BLUE = "#2B5C88"
LIGHT_GREY = "#F3F5F7"
DARK_GREY = "#3A3A3A"

CONFIDENCE_COLORS = {
    "High confidence": "#1B7837",
    "Medium confidence": "#E66101",
    "Serious caveat / manual review": "#B2182B",
}

LANE_COLORS = {
    "Clean Opportunity": "#1B7837",
    "Caveated Opportunity": "#E66101",
    "Breakthrough Build": "#5E3C99",
    "Long-Term Demographic Build": "#2C7FB8",
    "Monitor": "#BDBDBD",
}

PROCESS_COLORS = {
    # Full report labels
    "Conservative Legacy / Right-Adjacent Transition Terrain": "#2166AC",
    "Labour Stronghold Breakthrough Terrain": "#D73027",
    "Reform / Independent Disruption Terrain": "#7B3294",
    "Green / Liberal Democrat Non-Core Terrain": "#1A9850",
    # Short diagnostic labels actually present in the diagnostics CSV
    "Conservative Legacy / Right-Adjacent Transition": "#2166AC",
    "Labour Stronghold Breakthrough": "#D73027",
    "Reform / Independent Disruption": "#825c9d",
    "Green / Liberal Democrat Non-Core": "#1A9850",
}

PARTY_COLORS = {
    "lab": "#D73027",
    "con": "#2166AC",
    "ld": "#FDB863",
    "green": "#1A9850",
    "reform_ukip_brexit": "#7B3294",
    "independent": "#4D4D4D",
    "other": "#969696",
    "sdp": "#08306B",
}

plt.rcParams["figure.dpi"] = 140
plt.rcParams["savefig.dpi"] = 220
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 10
plt.rcParams["xtick.labelsize"] = 9
plt.rcParams["ytick.labelsize"] = 9

manifest_rows = []

def add_manifest(filename, asset_type, report_section, description, path):
    manifest_rows.append({
        "filename": filename,
        "asset_type": asset_type,
        "report_section": report_section,
        "description": description,
        "path": str(path),
        "created_at": datetime.now().isoformat(timespec="seconds"),
    })

## 26.3 Utility functions

In [86]:
def find_file(filename, required=True):
    direct_candidates = [PROCESSED_DIR / filename, NOTEBOOK_DIR / filename, PROJECT_DIR / filename]
    for path in direct_candidates:
        if path.exists():
            return path
    if PROCESSED_DIR.exists():
        matches = list(PROCESSED_DIR.rglob(filename))
        if matches:
            # Prefer highest suffix version if names differ only by folder timestamp; otherwise newest modified.
            matches = sorted(matches, key=lambda p: p.stat().st_mtime, reverse=True)
            return matches[0]
    # Useful when testing in ChatGPT sandbox or manually dropping files next to notebook.
    sandbox = Path("/mnt/data") / filename
    if sandbox.exists():
        return sandbox
    if required:
        raise FileNotFoundError(f"Could not find required file: {filename}")
    return None


def read_csv(filename, required=True):
    path = find_file(filename, required=required)
    if path is None:
        print("Optional missing:", filename)
        return None
    df = pd.read_csv(path, low_memory=False)
    print(f"Loaded {filename}: {df.shape} from {path}")
    return df


def save_table(df, filename, section, description, folder=TABLE_DIR):
    path = folder / filename
    df.to_csv(path, index=False)
    add_manifest(filename, "table_csv", section, description, path)
    print("Saved:", path)
    return path


def save_text(text, filename, section, description):
    path = TEXT_DIR / filename
    path.write_text(text, encoding="utf-8")
    add_manifest(filename, "text_md", section, description, path)
    print("Saved:", path)
    return path


def save_chart(fig, filename, section, description):
    path = CHART_DIR / filename
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    add_manifest(filename, "chart_png", section, description, path)
    print("Saved:", path)
    return path


def pct(x, digits=1):
    if pd.isna(x):
        return ""
    return f"{float(x) * 100:.{digits}f}%"


def score(x, digits=1):
    if pd.isna(x):
        return ""
    return f"{float(x):.{digits}f}"


def wrap_labels(labels, width=24):
    return ["\n".join(textwrap.wrap(str(label), width=width)) for label in labels]

def existing(df, cols):
    return [c for c in cols if c in df.columns]

## 26.4 Load corrected pre-Adam inputs

This notebook intentionally uses mixed suffixes:

- `v4` for caveat, party-transition and Yorkshire/reporting outputs;
- `v3` for corrected SDP validation outputs.

That reflects the actual corrected pipeline state.

In [87]:
headline = read_csv("pre_adam_headline_metrics_v3.csv")
notes_path = find_file("pre_adam_report_revision_notes_v4.md", required=False)
notes_text = notes_path.read_text(encoding="utf-8") if notes_path else ""

high_conf = read_csv("pre_adam_high_confidence_top100_v4.csv")
medium_conf = read_csv("pre_adam_medium_confidence_top100_v4.csv")
serious_caveat = read_csv("pre_adam_serious_caveat_top100_v4.csv", required=False)

party_diag = read_csv("pre_adam_party_transition_diagnostics_all_v4.csv")
party_council = read_csv("pre_adam_party_transition_summary_by_council_v4.csv")
party_labels = read_csv("pre_adam_party_process_label_notes_v4.csv")

sdp_count = read_csv("pre_adam_sdp_candidate_count_check_v3.csv")
sdp_tribe = read_csv("pre_adam_sdp_performance_by_tribe_v3.csv")
sdp_latest_party = read_csv("pre_adam_sdp_performance_by_latest_party_v3.csv")
sdp_highest = read_csv("pre_adam_sdp_highest_vote_share_cases_v3.csv")
sdp_year = read_csv("pre_adam_sdp_performance_by_year_v3.csv", required=False)
sdp_unmatched = read_csv("pre_adam_sdp_unmatched_rows_for_review_v3.csv", required=False)

yorks_summary = read_csv("pre_adam_yorkshire_case_study_summary_v4.csv")
yorks_wards = read_csv("pre_adam_yorkshire_sdp_case_study_wards_v4.csv")
yorks_tribe = read_csv("pre_adam_yorkshire_sdp_performance_by_tribe_v4.csv")
middleton = read_csv("pre_adam_middleton_park_case_study_profile_v4.csv")

print("Inputs loaded.")

Loaded pre_adam_headline_metrics_v3.csv: (1, 13) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\pre_adam_report_revision_v4\pre_adam_headline_metrics_v3.csv
Loaded pre_adam_high_confidence_top100_v4.csv: (671, 74) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\pre_adam_report_revision_v4\pre_adam_high_confidence_top100_v4.csv
Loaded pre_adam_medium_confidence_top100_v4.csv: (153, 74) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\pre_adam_report_revision_v4\pre_adam_medium_confidence_top100_v4.csv
Loaded pre_adam_serious_caveat_top100_v4.csv: (1, 74) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\pre_adam_report_revision_v4\pre_adam_serious_caveat_top100_v4.csv
Loaded pre_adam_party_transition_diagnostics_all_v4.csv: (825, 29) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\pre_adam_report_revision_v4\pre_adam_party_transition_diagnostics_all_v4.csv
Loaded pre_adam_party_transition_summary_by_council_v4.csv: (35,

## 26.5 Validation checks

These checks guard against the previous stale-source issues.

In [88]:
# Basic sanity checks.
metrics = headline.iloc[0].to_dict()
print(metrics)

assert int(metrics.get("sdp_rows_total", 0)) >= 150, "SDP validation source looks stale: expected ~171 rows."
assert int(metrics.get("sdp_rows_matched_to_model", 0)) >= 140, "Matched SDP rows unexpectedly low."
assert int(metrics.get("high_confidence_rows", 0)) == 671, "High confidence row count differs from expected 671."
assert int(metrics.get("medium_confidence_rows", 0)) == 153, "Medium confidence row count differs from expected 153."
assert int(metrics.get("serious_caveat_rows", 0)) == 1, "Serious caveat row count differs from expected 1."

# Party transition confidence should no longer be all serious/manual review.
if "report_confidence_band_v2" in party_diag.columns:
    print(party_diag["report_confidence_band_v2"].value_counts(dropna=False))
    assert (party_diag["report_confidence_band_v2"] == "Serious caveat / manual review").sum() < len(party_diag), "Party transition confidence still stale."

print("Validation checks passed.")

{'run_date': '2026-05-28T15:02:45', 'high_confidence_rows': 671, 'medium_confidence_rows': 153, 'serious_caveat_rows': 1, 'main_report_rows': 824, 'sdp_rows_total': 171, 'sdp_rows_matched_to_model': 153, 'sdp_unmatched_rows': 18, 'sdp_max_vote_share': 0.5075557234605214, 'yorkshire_case_study_rows': 53, 'yorkshire_max_sdp_vote_share': 0.5075557234605214, 'yorkshire_mean_sdp_vote_share': 0.0321224122676731, 'yorkshire_top_dominant_tribe': 'Post-Industrial Estates / Deprived Working Communities'}
report_confidence_band_v2
High confidence      671
Medium confidence    153
NaN                    1
Name: count, dtype: int64
Validation checks passed.


## 26.6 Headline report tables

In [89]:
headline_table = headline.copy()
# Add display-friendly fields.
headline_display = pd.DataFrame([
    ["High-confidence North West rows", int(metrics.get("high_confidence_rows", 0))],
    ["Medium-confidence North West rows", int(metrics.get("medium_confidence_rows", 0))],
    ["Serious caveat/manual-review rows", int(metrics.get("serious_caveat_rows", 0))],
    ["Main report rows", int(metrics.get("main_report_rows", 0))],
    ["SDP validation rows", int(metrics.get("sdp_rows_total", 0))],
    ["SDP rows matched to model", int(metrics.get("sdp_rows_matched_to_model", 0))],
    ["SDP unmatched rows", int(metrics.get("sdp_unmatched_rows", 0))],
    ["Max observed SDP vote share", pct(metrics.get("sdp_max_vote_share", np.nan))],
    ["Yorkshire case-study rows", int(metrics.get("yorkshire_case_study_rows", 0))],
    ["Yorkshire max SDP vote share", pct(metrics.get("yorkshire_max_sdp_vote_share", np.nan))],
    ["Yorkshire top dominant tribe", metrics.get("yorkshire_top_dominant_tribe", "")],
], columns=["Metric", "Value"])

save_table(headline_table, "headline_metrics_raw_v1.csv", "Executive Summary", "Raw headline metrics from the corrected pre-Adam pack.")
save_table(headline_display, "headline_metrics_report_table_v1.csv", "Executive Summary", "Display-friendly headline metrics table.")
headline_display

Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_pre_adam_v2\tables\headline_metrics_raw_v1.csv
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_pre_adam_v2\tables\headline_metrics_report_table_v1.csv


,Metric,Value
0,High-confidence North West rows,671
1,Medium-confidence North West rows,153
2,Serious caveat/manual-review rows,1
3,Main report rows,824
4,SDP validation rows,171
5,SDP rows matched to model,153
6,SDP unmatched rows,18
7,Max observed SDP vote share,50.8%
8,Yorkshire case-study rows,53
9,Yorkshire max SDP vote share,50.8%


In [90]:
# Core report tables.
high_cols = existing(high_conf, [
    "LAD25NM", "WD25NM", "initial_watchlist_score", "revised_strategic_lane_v2",
    "report_confidence_band_v2", "dominant_cluster_name", "second_cluster_name",
    "latest_election_top_party_bucket", "demographic_relevance_score", "electoral_opportunity_score",
    "political_openness_score", "breakthrough_complacency_score"
])
medium_cols = existing(medium_conf, high_cols + ["report_caveat_summary_v2", "electoral_evidence_type_v2"])

save_table(high_conf[high_cols].head(25), "top_25_high_confidence_rows_report_table_v1.csv", "North West Findings", "Top high-confidence North West rows for the main report.")
save_table(medium_conf[medium_cols].head(25), "top_25_medium_confidence_rows_report_table_v1.csv", "North West Findings", "Top medium-confidence North West rows for the report with caveat note.")

if serious_caveat is not None and len(serious_caveat):
    save_table(serious_caveat[existing(serious_caveat, medium_cols)].head(10), "serious_caveat_rows_report_table_v1.csv", "Caveats", "Serious caveat/manual-review rows for appendix only.")

# Party labels and summaries.
save_table(party_labels, "party_process_label_notes_report_table_v1.csv", "Party Process Diagnostics", "Definitions and report use for party-process labels.")
party_council_cols = existing(party_council, [
    "LAD25NM", "wards", "mean_conservative_transition_score", "mean_labour_stronghold_breakthrough_score",
    "mean_reform_independent_disruption_score", "mean_green_ld_noncore_score", "top_model_score"
])
save_table(party_council[party_council_cols].head(25), "party_transition_summary_by_council_report_table_v1.csv", "Party Process Diagnostics", "Council-level party-process diagnostic summary.")

# SDP validation report tables.
save_table(sdp_count, "sdp_candidate_count_check_report_table_v1.csv", "SDP Validation", "Expected versus observed SDP candidate row counts.")
save_table(sdp_tribe.head(10), "sdp_performance_by_tribe_report_table_v1.csv", "SDP Validation", "SDP performance by dominant tribe.")
save_table(sdp_latest_party.head(10), "sdp_performance_by_latest_party_report_table_v1.csv", "SDP Validation", "SDP performance by latest top party.")
save_table(sdp_highest.head(25), "sdp_highest_vote_share_cases_report_table_v1.csv", "SDP Validation", "Highest observed SDP vote-share cases.")

# Yorkshire tables.
save_table(yorks_summary, "yorkshire_case_study_summary_report_table_v1.csv", "Yorkshire Case Study", "Headline Yorkshire case-study metrics.")
save_table(yorks_wards.head(25), "yorkshire_case_study_top_wards_report_table_v1.csv", "Yorkshire Case Study", "Top Yorkshire SDP case-study wards.")
save_table(yorks_tribe.head(10), "yorkshire_sdp_performance_by_tribe_report_table_v1.csv", "Yorkshire Case Study", "Yorkshire SDP performance by dominant tribe.")
save_table(middleton, "middleton_park_case_study_profile_report_table_v1.csv", "Middleton Park", "Full Middleton Park case-study profile.")

Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_pre_adam_v2\tables\top_25_high_confidence_rows_report_table_v1.csv
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_pre_adam_v2\tables\top_25_medium_confidence_rows_report_table_v1.csv
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_pre_adam_v2\tables\serious_caveat_rows_report_table_v1.csv
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_pre_adam_v2\tables\party_process_label_notes_report_table_v1.csv
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_pre_adam_v2\tables\party_transition_summary_by_council_report_table_v1.csv
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_pre_adam_v2\tables\sdp_candidate_count_check_report_table_v1.csv
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_pre_adam_v2\tables\sdp_performance_by_tribe_report_tab

WindowsPath('c:/Users/keena/Documents/Electoral_Tribes/data/processed/report_assets_pre_adam_v2/tables/middleton_park_case_study_profile_report_table_v1.csv')

## 26.7 Appendix tables

These are full export tables for the appendix or internal review, not main-report tables.

In [91]:
appendix_tables = {
    "appendix_high_confidence_top100_v1.csv": high_conf,
    "appendix_medium_confidence_top100_v1.csv": medium_conf,
    "appendix_serious_caveat_rows_v1.csv": serious_caveat if serious_caveat is not None else pd.DataFrame(),
    "appendix_party_transition_diagnostics_all_v1.csv": party_diag,
    "appendix_party_transition_summary_by_council_v1.csv": party_council,
    "appendix_sdp_highest_vote_share_cases_v1.csv": sdp_highest,
    "appendix_sdp_unmatched_rows_for_review_v1.csv": sdp_unmatched if sdp_unmatched is not None else pd.DataFrame(),
    "appendix_yorkshire_sdp_case_study_wards_v1.csv": yorks_wards,
    "appendix_middleton_park_case_study_profile_v1.csv": middleton,
}

for filename, df in appendix_tables.items():
    if df is not None and len(df) > 0:
        save_table(df, filename, "Appendix", f"Full appendix table: {filename}", folder=APPENDIX_DIR)

Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_pre_adam_v2\appendices\appendix_high_confidence_top100_v1.csv
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_pre_adam_v2\appendices\appendix_medium_confidence_top100_v1.csv
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_pre_adam_v2\appendices\appendix_serious_caveat_rows_v1.csv
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_pre_adam_v2\appendices\appendix_party_transition_diagnostics_all_v1.csv
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_pre_adam_v2\appendices\appendix_party_transition_summary_by_council_v1.csv
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_pre_adam_v2\appendices\appendix_sdp_highest_vote_share_cases_v1.csv
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_pre_adam_v2\appendices\appendix_sdp_unmatched_rows_for

## 26.8 Charts for report

In [92]:

# -------------------------------------------------------------------
# Chart helpers
# -------------------------------------------------------------------
def as_numeric(series, default=0):
    return pd.to_numeric(series, errors="coerce").fillna(default)


def as_label(series, default="Unknown"):
    return series.astype("string").fillna(default).astype(str)


# Chart 1: confidence bands.
conf = pd.DataFrame({
    "Confidence band": ["High confidence", "Medium confidence", "Serious caveat / manual review"],
    "Rows": [metrics.get("high_confidence_rows", 0), metrics.get("medium_confidence_rows", 0), metrics.get("serious_caveat_rows", 0)]
})
conf["Rows"] = as_numeric(conf["Rows"])
fig, ax = plt.subplots(figsize=(8.5, 4.5))
ax.bar(conf["Confidence band"], conf["Rows"], color=[CONFIDENCE_COLORS[x] for x in conf["Confidence band"]])
ax.set_title("North West model rows by confidence band", color=NAVY, weight="bold")
ax.set_ylabel("Rows")
ax.set_xticklabels(wrap_labels(conf["Confidence band"], 18))
ax.spines[["top", "right"]].set_visible(False)
for i, v in enumerate(conf["Rows"]):
    ax.text(i, v + max(conf["Rows"]) * 0.01, str(int(v)), ha="center", va="bottom")
save_chart(fig, "chart_01_confidence_bands_v1.png", "Executive Summary", "North West model rows by report confidence band.")

# Chart 2: expected vs observed SDP rows.
if {"expected_user_provided", "observed_sdp_candidate_rows"}.issubset(sdp_count.columns):
    tmp = sdp_count.copy()
    tmp["expected_user_provided"] = as_numeric(tmp["expected_user_provided"])
    tmp["observed_sdp_candidate_rows"] = as_numeric(tmp["observed_sdp_candidate_rows"])
    x = np.arange(len(tmp))
    width = 0.38
    fig, ax = plt.subplots(figsize=(8.5, 4.8))
    ax.bar(x - width/2, tmp["expected_user_provided"], width, label="Expected", color="#BDBDBD")
    ax.bar(x + width/2, tmp["observed_sdp_candidate_rows"], width, label="Observed", color=NAVY)
    ax.set_xticks(x)
    ax.set_xticklabels(as_label(tmp["election_year"]))
    ax.set_title("SDP candidate rows: expected vs observed", color=NAVY, weight="bold")
    ax.set_ylabel("Candidate rows")
    ax.legend(frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    save_chart(fig, "chart_02_sdp_candidate_rows_expected_vs_observed_v1.png", "SDP Validation", "Expected versus observed SDP candidate rows by year.")

# Chart 3: SDP performance by dominant tribe.
if "dominant_cluster_name" in sdp_tribe.columns and "mean_sdp_vote_share" in sdp_tribe.columns:
    st = sdp_tribe.copy()
    st["dominant_cluster_name"] = as_label(st["dominant_cluster_name"])
    st["mean_sdp_vote_share"] = as_numeric(st["mean_sdp_vote_share"])
    st = st.sort_values("mean_sdp_vote_share", ascending=True).tail(8)
    fig, ax = plt.subplots(figsize=(9, 5.5))
    ax.barh(st["dominant_cluster_name"], st["mean_sdp_vote_share"] * 100, color=MID_BLUE)
    ax.set_title("Mean SDP vote share by dominant tribe", color=NAVY, weight="bold")
    ax.set_xlabel("Mean SDP vote share (%)")
    ax.spines[["top", "right"]].set_visible(False)
    save_chart(fig, "chart_03_sdp_mean_vote_share_by_tribe_v1.png", "SDP Validation", "Mean SDP vote share by dominant tribe.")

# Chart 4: SDP highest cases.
if "sdp_vote_share_effective" in sdp_highest.columns:
    top = sdp_highest.copy()
    top["sdp_vote_share_effective"] = as_numeric(top["sdp_vote_share_effective"])
    top = top.sort_values("sdp_vote_share_effective", ascending=False).head(15)
    label_col = "WD25NM_model" if "WD25NM_model" in top.columns else ("WD25NM" if "WD25NM" in top.columns else "ward_name")
    labels = as_label(top[label_col])
    fig, ax = plt.subplots(figsize=(9, 5.5))
    ax.barh(labels[::-1], top["sdp_vote_share_effective"][::-1] * 100, color="#08306B")
    ax.set_title("Highest observed SDP vote-share cases", color=NAVY, weight="bold")
    ax.set_xlabel("SDP vote share (%)")
    ax.spines[["top", "right"]].set_visible(False)
    save_chart(fig, "chart_04_sdp_highest_vote_share_cases_v1.png", "SDP Validation", "Highest observed SDP vote-share cases.")

# Chart 5: Yorkshire case-study wards.
if "max_sdp_vote_share" in yorks_wards.columns:
    top = yorks_wards.copy()
    top["max_sdp_vote_share"] = as_numeric(top["max_sdp_vote_share"])
    top = top.sort_values("max_sdp_vote_share", ascending=False).head(15)
    label_col = "WD25NM" if "WD25NM" in top.columns else ("ward_name" if "ward_name" in top.columns else top.columns[0])
    labels = as_label(top[label_col])
    fig, ax = plt.subplots(figsize=(9, 5.5))
    ax.barh(labels[::-1], top["max_sdp_vote_share"][::-1] * 100, color="#5E3C99")
    ax.set_title("Yorkshire case study: strongest SDP ward performances", color=NAVY, weight="bold")
    ax.set_xlabel("Maximum SDP vote share (%)")
    ax.spines[["top", "right"]].set_visible(False)
    save_chart(fig, "chart_05_yorkshire_strongest_sdp_wards_v1.png", "Yorkshire Case Study", "Strongest Yorkshire SDP ward performances by maximum vote share.")

# Chart 6: Yorkshire performance by tribe.
if "dominant_cluster_name" in yorks_tribe.columns and "mean_sdp_vote_share" in yorks_tribe.columns:
    yt = yorks_tribe.copy()
    yt["dominant_cluster_name"] = as_label(yt["dominant_cluster_name"])
    yt["mean_sdp_vote_share"] = as_numeric(yt["mean_sdp_vote_share"])
    yt = yt.sort_values("mean_sdp_vote_share", ascending=True)
    fig, ax = plt.subplots(figsize=(9, 5.5))
    ax.barh(yt["dominant_cluster_name"], yt["mean_sdp_vote_share"] * 100, color="#762A83")
    ax.set_title("Yorkshire SDP mean vote share by dominant tribe", color=NAVY, weight="bold")
    ax.set_xlabel("Mean SDP vote share (%)")
    ax.spines[["top", "right"]].set_visible(False)
    save_chart(fig, "chart_06_yorkshire_sdp_vote_share_by_tribe_v1.png", "Yorkshire Case Study", "Yorkshire SDP performance by dominant tribe.")

# Chart 7: party-process diagnostic top councils.
process_cols = [
    "mean_conservative_transition_score",
    "mean_labour_stronghold_breakthrough_score",
    "mean_reform_independent_disruption_score",
]
if all(c in party_council.columns for c in process_cols):
    pc = party_council.copy()
    for col in process_cols:
        pc[col] = as_numeric(pc[col])
    if "top_model_score" in pc.columns:
        pc["top_model_score"] = as_numeric(pc["top_model_score"])
        pc = pc.sort_values("top_model_score", ascending=False).head(15)
    else:
        pc = pc.sort_values(process_cols, ascending=False).head(15)
    council_labels = as_label(pc["LAD25NM"] if "LAD25NM" in pc.columns else pc.iloc[:, 0])
    fig, ax = plt.subplots(figsize=(10, 6))
    bottom = np.zeros(len(pc))
    colours = ["#2166AC", "#D73027", "#7B3294"]
    labels = ["Conservative legacy/right-adjacent", "Labour stronghold breakthrough", "Reform/Independent disruption"]
    for col, label, colour in zip(process_cols, labels, colours):
        values = pc[col].to_numpy(dtype=float)
        ax.barh(council_labels, values, left=bottom, label=label, color=colour)
        bottom += values
    ax.set_title("Party-process diagnostic scores by leading councils", color=NAVY, weight="bold")
    ax.set_xlabel("Diagnostic score (stacked)")
    ax.invert_yaxis()
    ax.legend(frameon=False, fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)
    save_chart(fig, "chart_07_party_process_scores_by_council_v1.png", "Party Process Diagnostics", "Party-process diagnostic scores by leading councils.")


Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_pre_adam_v2\charts\chart_01_confidence_bands_v1.png
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_pre_adam_v2\charts\chart_02_sdp_candidate_rows_expected_vs_observed_v1.png
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_pre_adam_v2\charts\chart_03_sdp_mean_vote_share_by_tribe_v1.png
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_pre_adam_v2\charts\chart_04_sdp_highest_vote_share_cases_v1.png
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_pre_adam_v2\charts\chart_05_yorkshire_strongest_sdp_wards_v1.png
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_pre_adam_v2\charts\chart_06_yorkshire_sdp_vote_share_by_tribe_v1.png
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_pre_adam_v2\charts\chart_07_party_process_scores_by_council_v1.pn

## 26.9 Optional map assets

This section generates maps only if a WD25 ward boundary file exists in `data/geography/boundaries`.

Required boundary key: `WD25CD`.

In [93]:
def find_boundary_file():
    if not BOUNDARY_DIR.exists():
        return None
    candidates = []
    for pattern in ["*.gpkg", "*.shp", "*.geojson", "*.json"]:
        candidates.extend(sorted(BOUNDARY_DIR.glob(pattern)))
    preferred = [p for p in candidates if "ward" in p.name.lower() or "wd25" in p.name.lower()]
    return preferred[0] if preferred else (candidates[0] if candidates else None)

boundary_file = find_boundary_file()
print("Boundary file:", boundary_file)

if boundary_file is not None:
    try:
        import geopandas as gpd
        MAPS_AVAILABLE = True
    except Exception as e:
        print("geopandas unavailable; skipping maps.", e)
        MAPS_AVAILABLE = False
else:
    MAPS_AVAILABLE = False

print("Maps available:", MAPS_AVAILABLE)

Boundary file: c:\Users\keena\Documents\Electoral_Tribes\data\geography\Wards_May_2025_Boundaries_UK_BGC.gpkg
Maps available: True


In [94]:
if MAPS_AVAILABLE:
    wards = gpd.read_file(boundary_file)
    if "WD25CD" not in wards.columns:
        matches = [c for c in wards.columns if c.upper() == "WD25CD" or "WD25CD" in c.upper()]
        if matches:
            wards = wards.rename(columns={matches[0]: "WD25CD"})
        else:
            raise ValueError("Boundary file must contain WD25CD or recognisable equivalent.")
    wards["WD25CD"] = wards["WD25CD"].astype(str).str.strip()
    map_df = party_diag.copy()
    map_df["WD25CD"] = map_df["WD25CD"].astype(str).str.strip()
    gdf = wards.merge(map_df, on="WD25CD", how="inner")
    print("Mapped rows:", len(gdf))
    try:
        gdf = gdf.to_crs(27700)
    except Exception:
        pass

    # Map 1: confidence band.
    fig, ax = plt.subplots(figsize=(10, 11))
    gdf["_confidence_colour"] = gdf["report_confidence_band_v2"].map(CONFIDENCE_COLORS).fillna("#D9D9D9")
    gdf.plot(ax=ax, color=gdf["_confidence_colour"], linewidth=0.08, edgecolor="white")
    ax.set_axis_off()
    ax.set_title("North West report confidence bands", color=NAVY, weight="bold", fontsize=16)
    handles = [Patch(facecolor=CONFIDENCE_COLORS[k], label=k) for k in CONFIDENCE_COLORS if k in set(gdf["report_confidence_band_v2"].dropna())]
    ax.legend(handles=handles, loc="lower left", frameon=True, fontsize=8)
    p = MAP_DIR / "map_01_report_confidence_bands_v1.png"
    fig.savefig(p, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    add_manifest(p.name, "map_png", "Confidence Bands", "North West wards by report confidence band.", p)

    # Map 2: primary party-transition diagnostic.
    # The diagnostics file may use short labels, while the report uses fuller labels.
    process_label_col = "primary_party_transition_diagnostic"
    gdf[process_label_col] = gdf[process_label_col].astype("string").fillna("Unknown").astype(str).str.strip()
    gdf["_process_colour"] = gdf[process_label_col].map(PROCESS_COLORS).fillna("#D9D9D9")
    print("Party-process diagnostic values on map:")
    print(gdf[process_label_col].value_counts(dropna=False))

    fig, ax = plt.subplots(figsize=(10, 11))
    gdf.plot(ax=ax, color=gdf["_process_colour"], linewidth=0.08, edgecolor="white")
    ax.set_axis_off()
    ax.set_title("North West party-process diagnostics", color=NAVY, weight="bold", fontsize=16)
    present_processes = [x for x in gdf[process_label_col].dropna().unique().tolist() if x in PROCESS_COLORS]
    handles = [Patch(facecolor=PROCESS_COLORS[k], label=k) for k in present_processes]
    if handles:
        ax.legend(handles=handles, loc="lower left", frameon=True, fontsize=8)
    else:
        print("Warning: no party-process legend handles created. Check label values and PROCESS_COLORS.")
    p = MAP_DIR / "map_02_party_process_diagnostics_v1.png"
    fig.savefig(p, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    add_manifest(p.name, "map_png", "Party Process Diagnostics", "North West wards by primary party-process diagnostic.", p)

    # Map 3: model score.
    fig, ax = plt.subplots(figsize=(10, 11))
    gdf.plot(ax=ax, column="initial_watchlist_score", cmap="YlOrRd", legend=True, linewidth=0.08, edgecolor="white", missing_kwds={"color": "#F0F0F0"})
    ax.set_axis_off()
    ax.set_title("North West structural opportunity score", color=NAVY, weight="bold", fontsize=16)
    p = MAP_DIR / "map_03_initial_watchlist_score_v1.png"
    fig.savefig(p, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    add_manifest(p.name, "map_png", "North West Findings", "North West wards by model / structural opportunity score.", p)
else:
    print("Skipping maps.")

Mapped rows: 825
Party-process diagnostic values on map:
primary_party_transition_diagnostic
Labour Stronghold Breakthrough     417
Reform / Independent Disruption    408
Name: count, dtype: int64


## 26.10 Draft report text snippets

These markdown snippets can be copied directly into the report draft.

In [95]:
executive_summary_text = f"""
# Executive summary draft

The North West structural model should be presented as a breakthrough and organisational build model, not a council-control model and not a final target-seat list. It identifies structurally favourable wards for selective breakthrough, organisational build and further political review.

The corrected pre-Adam pack contains {int(metrics.get('high_confidence_rows'))} high-confidence rows, {int(metrics.get('medium_confidence_rows'))} medium-confidence rows and {int(metrics.get('serious_caveat_rows'))} serious/manual-review row. This means technical mapping issues no longer need to be presented as strategic warnings; most medium-confidence rows are still usable with clear notes.

The SDP validation layer now contains {int(metrics.get('sdp_rows_total'))} rows, of which {int(metrics.get('sdp_rows_matched_to_model'))} are matched to the model. The maximum observed SDP vote share is {pct(metrics.get('sdp_max_vote_share'))}. The strongest empirical validation comes from Yorkshire, where actual high SDP performance is concentrated in Post-Industrial / working-community wards, especially where paired with Settled Working Families.

Model v1 does not forecast council control, vote share or seat totals. A separate vote-share and seat-simulation model would be required for that purpose.
""".strip()

sdp_validation_text = """
# SDP validation draft

The SDP campaign evidence is the first true validation layer for the model. It tests whether the structurally favourable wards identified by the demographic and electoral model resemble places where the SDP has already demonstrated traction.

The current evidence supports a clear but bounded conclusion: the strongest observed SDP performances are concentrated in Post-Industrial / working-community wards, with Settled Working Families appearing as an important adjacent or secondary terrain. Middleton Park is the proof case, but Dearne South and Wath suggest the pattern is not unique to one ward.

Conservative-adjacent terrain remains plausible but less strongly validated. It should therefore be labelled as Conservative Legacy / Right-Adjacent Transition Terrain rather than treated as proven current Conservative-held opportunity.
""".strip()

yorkshire_text = """
# Yorkshire case study draft

The Yorkshire case study is the strongest empirical validation section of the report. Middleton Park shows exceptional SDP performance, but the wider lesson is not simply that “Leeds works”. Most other Leeds wards show weak SDP performance. The actual evidence is narrower and more useful: Middleton Park works as a specific Post-Industrial + Settled Working Families breakthrough case.

Dearne South and Wath provide secondary validation. They share similar dominant and secondary tribe structure and show materially stronger SDP performance than most other contested wards. This supports the hypothesis that the most promising observed SDP terrain is Post-Industrial / working-community geography, especially when combined with local candidate credibility and repeated campaigning.
""".strip()

control_model_text = """
# Control model limitation draft

This report does not identify a clear council-control pathway. The current model is not designed to forecast vote share, seat totals, councillor numbers, opposition group status, balance of power or council control. It is a structural breakthrough and organisational build model.

A separate model would be required to estimate control scenarios. That model would need expected vote shares, adjacent-voter conversion assumptions, candidate effects, local campaign capacity, opposition fragmentation and turnout efficiency.
""".strip()

save_text(executive_summary_text, "draft_executive_summary_v1.md", "Executive Summary", "Draft executive summary text.")
save_text(sdp_validation_text, "draft_sdp_validation_section_v1.md", "SDP Validation", "Draft SDP validation text.")
save_text(yorkshire_text, "draft_yorkshire_case_study_section_v1.md", "Yorkshire Case Study", "Draft Yorkshire case study text.")
save_text(control_model_text, "draft_control_model_limitation_section_v1.md", "Caveats", "Draft limitation/control-model text.")

Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_pre_adam_v2\text\draft_executive_summary_v1.md
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_pre_adam_v2\text\draft_sdp_validation_section_v1.md
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_pre_adam_v2\text\draft_yorkshire_case_study_section_v1.md
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_pre_adam_v2\text\draft_control_model_limitation_section_v1.md


WindowsPath('c:/Users/keena/Documents/Electoral_Tribes/data/processed/report_assets_pre_adam_v2/text/draft_control_model_limitation_section_v1.md')

## 26.11 Asset manifest

In [96]:
manifest = pd.DataFrame(manifest_rows)
manifest_path = MANIFEST_DIR / "pre_adam_report_asset_manifest_v1.csv"
manifest.to_csv(manifest_path, index=False)
print("Saved manifest:", manifest_path)
manifest

Saved manifest: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_pre_adam_v2\manifest\pre_adam_report_asset_manifest_v1.csv


,filename,asset_type,report_section,description,path,created_at
0,headline_metrics_raw_v1.csv,table_csv,Executive Summary,Raw headline metrics from the corrected pre-Ad...,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-05-28T15:33:21
1,headline_metrics_report_table_v1.csv,table_csv,Executive Summary,Display-friendly headline metrics table.,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-05-28T15:33:21
2,top_25_high_confidence_rows_report_table_v1.csv,table_csv,North West Findings,Top high-confidence North West rows for the ma...,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-05-28T15:33:21
3,top_25_medium_confidence_rows_report_table_v1.csv,table_csv,North West Findings,Top medium-confidence North West rows for the ...,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-05-28T15:33:21
4,serious_caveat_rows_report_table_v1.csv,table_csv,Caveats,Serious caveat/manual-review rows for appendix...,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-05-28T15:33:21
5,party_process_label_notes_report_table_v1.csv,table_csv,Party Process Diagnostics,Definitions and report use for party-process l...,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-05-28T15:33:21
6,party_transition_summary_by_council_report_tab...,table_csv,Party Process Diagnostics,Council-level party-process diagnostic summary.,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-05-28T15:33:21
7,sdp_candidate_count_check_report_table_v1.csv,table_csv,SDP Validation,Expected versus observed SDP candidate row cou...,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-05-28T15:33:21
8,sdp_performance_by_tribe_report_table_v1.csv,table_csv,SDP Validation,SDP performance by dominant tribe.,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-05-28T15:33:21
9,sdp_performance_by_latest_party_report_table_v...,table_csv,SDP Validation,SDP performance by latest top party.,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-05-28T15:33:21


## 26.12 Final report assembly checklist

Use the generated assets to revise the draft report in this order:

1. Replace old caveat language with confidence-band language.
2. Insert the SDP validation section.
3. Insert the Yorkshire case study section.
4. Insert party-process diagnostic definitions and tables.
5. Replace any “target list” language with “watchlist”, “breakthrough geography” or “build geography”.
6. Keep council-control claims out of the report.